# FingerBall Stage 2H — Manual-guided Hand Landmark Annotation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2H_Manual_Guided_Annotation.ipynb)

**目的：** リリース前後で背景やボールに重なって誤検出されたMediaPipe 21点を、元動画の座標系でドラッグ修正します。元データは保持し、修正点・遮蔽点・補間点の由来をCSVとJSONに保存します。

**主な操作：** フレームを選択 → 近くの点をドラッグ → 必要に応じて「遮蔽・不明」に設定 → キーフレームを保存 → ほかのフレームを編集。対象フレームを飛ばす・評価用フレームを確保することも可能です。

**重要な限界：** 修正するのは**2D投影座標だけ**です。MediaPipeの相対Zや3D関節角を「正しくなった」とは扱いません。遮蔽点を勝手に真値化しません。

## 0. セットアップ（Colab標準HTML Canvasエディタ：ipympl不要）

In [ ]:
%pip -q install "opencv-python-headless>=4.8" "pandas>=2" "matplotlib>=3.8"

In [ ]:
from google.colab import drive, output
drive.mount('/content/drive')
from pathlib import Path
import json, datetime, zipfile, base64
import cv2, numpy as np, pandas as pd
from IPython.display import display, HTML

## 1. 入力・出力の設定

元動画を使用してください。Stage 2Dの`release_zoom_175_210.mp4`は画像座標が変わっているため、このNotebookのアノテーションには使用できません。元動画の場所が見つからなければ`VIDEO_PATH`を明示指定します。

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d'
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
CSV_PATH=''                       # 空ならStage 2Dの21点CSV
VIDEO_PATH=''                     # 空ならMyDrive/母指回内トラッキング配下から元動画を探索
FIRST_FRAME,LAST_FRAME=185,198
KEY_FRAMES=[187,189,190,191,193,196]
THROWING_HAND='Left'              # 左右は確認のうえ手動指定
CROP_SIZE=300                     # 元動画の画像ピクセル、表示の一辺
AUTO_SAVE=True
assert THROWING_HAND in ('Left','Right')
project=Path(PROJECT_PATH)
if not project.is_dir():raise FileNotFoundError(project)
if CSV_PATH:
    csv_path=Path(CSV_PATH)
else:
    base=project/f'FingerBall_v2_{SOURCE_STAGE}_results'
    csv_path=base/VIDEO_STEM/'hand_21_landmarks_long.csv'
    if not csv_path.is_file():
        hits=sorted(base.glob('*/hand_21_landmarks_long.csv'))
        if not hits:raise FileNotFoundError('CSV_PATHを指定してください。21点CSVがありません')
        csv_path=hits[0]
if not csv_path.is_file():raise FileNotFoundError(csv_path)
if VIDEO_PATH:
    video_path=Path(VIDEO_PATH)
else:
    candidates=[p for p in project.rglob('*') if p.is_file()
       and p.suffix.lower() in ('.mp4','.mov','.avi')
       and VIDEO_STEM in p.stem and 'zoom' not in p.stem.lower()
       and 'annotated' not in p.stem.lower()]
    candidates=sorted(candidates,key=lambda p:(p.name!=VIDEO_STEM+'.mp4',len(p.parts)))
    if not candidates:
        raise FileNotFoundError('元動画が見つかりません。VIDEO_PATHに元動画の絶対パスを指定してください')
    video_path=candidates[0]
if not video_path.is_file():raise FileNotFoundError(video_path)
if any(t in video_path.stem.lower() for t in ('zoom','annotated','comparison')):
    raise ValueError('座標の異なる加工動画は使用できません。元動画VIDEO_PATHを指定してください')
out_dir=project/'FingerBall_v2_stage2h_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
state_file=out_dir/'manual_annotation_session.json'
print('CSV:',csv_path)
print('ORIGINAL video:',video_path)
print('Saving into:',out_dir)

## 2. 動画とLandmarksの読み込み・座標一致チェック

In [ ]:
source_df=pd.read_csv(csv_path)
required={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if required-set(source_df.columns):
    raise ValueError('CSV必須列不足: '+str(required-set(source_df.columns)))
source_df['frame']=pd.to_numeric(source_df.frame,errors='raise').astype(int)
source_df['landmark_id']=pd.to_numeric(source_df.landmark_id,errors='raise').astype(int)
if source_df.duplicated(['frame','landmark_id']).any():
    raise ValueError('frame/landmark_idに重複があります')
for c in ['x_px','y_px','z_relative','detected']:
    source_df[c]=pd.to_numeric(source_df[c],errors='coerce')
cap=cv2.VideoCapture(str(video_path))
if not cap.isOpened():raise RuntimeError('元動画をOpenCVで開けません')
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
if not (0<=FIRST_FRAME<=LAST_FRAME<TOTAL):raise ValueError('指定フレーム範囲が動画外です')
for norm,pix,extent in [('x_norm','x_px',W),('y_norm','y_px',H)]:
    if norm in source_df:
        good=source_df[[norm,pix]].dropna()
        good=good[good[norm].abs()>1e-4]
        if len(good)>10:
            est=float(np.median(good[pix]/good[norm]))
            if abs(est-extent)>3:
                cap.release()
                raise ValueError(f'CSVの画像寸法は約{est:.1f}ですが、指定動画は{extent}です。加工動画ではなく元動画を確認')
FRAME_IDS=list(range(FIRST_FRAME,LAST_FRAME+1))
def get_video_frame(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
    ok,im=cap.read()
    if not ok:raise RuntimeError(f'frame {f} の読み込み失敗')
    return cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
original={};edits={};statuses={}
for f in FRAME_IDS:
    group=source_df[(source_df.frame==f)&source_df.landmark_id.between(0,20)].sort_values('landmark_id')
    arr=np.full((21,2),np.nan,float)
    if len(group)==21 and group.landmark_id.tolist()==list(range(21)):
        p=group[['x_px','y_px']].to_numpy(float)
        valid=np.isfinite(p).all(axis=1)&group.detected.fillna(0).to_numpy().astype(bool)
        arr[valid]=p[valid]
    original[f]=arr.copy()
    edits[f]=arr.copy()
    statuses[f]=['ai' if np.isfinite(arr[i]).all() else 'missing' for i in range(21)]
if state_file.is_file():
    saved=json.loads(state_file.read_text(encoding='utf-8'))
    if (saved.get('video_filename')==video_path.name and
        saved.get('csv_path')==str(csv_path) and saved.get('resolution')==[W,H]):
        for key,item in saved.get('frames',{}).items():
            f=int(key)
            if f not in edits:continue
            p=np.array(item['coords'],dtype=object)
            edits[f]=np.array([[np.nan if v is None else float(v) for v in pair] for pair in p],float)
            statuses[f]=list(item['statuses'])
        print('前回の修正セッションを復元しました')
    else:print('既存の保存ファイルは別動画/別CSVです。上書きしません（保存先を変更してください）')
print(f'Video: {W}x{H} / {TOTAL} frames / {FPS:.3f}fps')
print('Frame range:',FRAME_IDS[0],FRAME_IDS[-1])

## 3. ブラウザ上の21点アノテーションGUI（Colab Canvas版）

**Matplotlib/ipymplは使用しません。** Colab内のCanvasで直接クリック・ドラッグして修正できます。画像が表示されたら、フレーム選択と拡大率を調整し、関節点をドラッグします。「配置」モードでは選択した点を画像クリックで設置。「遮蔽」は位置を欠損にします。

編集内容は **[Driveに保存]** ボタンで即時保存され、各フレーム切替でも同期します。Colabを再接続しても保存済みJSONを読み込みます。

In [ ]:
NAMES=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
       'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
       'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
       'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
       'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
FINGERS={'Thumb':[0,1,2,3,4],'Index':[0,5,6,7,8],
         'Middle':[0,9,10,11,12],'Ring':[0,13,14,15,16],
         'Little':[0,17,18,19,20]}
COLORS={'Thumb':'#f59e0b','Index':'#06b6d4','Middle':'#d946ef',
        'Ring':'#9ca3af','Little':'#64748b'}
def save_session():
    payload={'version':'stage2h-canvas-v2',
      'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'video_filename':video_path.name,'csv_path':str(csv_path),
      'resolution':[W,H],'hand':THROWING_HAND,
      'frames':{str(f):{
       'coords':[[None if not np.isfinite(x) else float(x) for x in pt]
                 for pt in edits[f]],
       'statuses':list(statuses[f])} for f in FRAME_IDS}}
    tmp=state_file.with_suffix('.tmp')
    tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(state_file)
def receive_annotation(payload):
    frame=int(payload['frame'])
    if frame not in edits:raise ValueError('Unrecognized frame')
    coords=payload['coords']
    labels=payload['statuses']
    if len(coords)!=21 or len(labels)!=21:raise ValueError('Expected 21 landmarks')
    allowed={'ai','missing','manual_visible','occluded','interpolated'}
    if any(v not in allowed for v in labels):raise ValueError('Unknown annotation status')
    arr=np.array([[np.nan if a is None else float(a) for a in p] for p in coords])
    if arr.shape!=(21,2):raise ValueError('Invalid point dimensions')
    if np.isinf(arr).any():raise ValueError('Infinite positions are invalid')
    if any(np.isfinite(pt).any() and not np.isfinite(pt).all() for pt in arr):
        raise ValueError('Partial coordinate is invalid')
    if np.any((arr[np.isfinite(arr)]<0)):raise ValueError('Coordinates below zero')
    finite=np.isfinite(arr).all(axis=1)
    if finite.any() and (np.any(arr[finite,0]>=W) or np.any(arr[finite,1]>=H)):
        raise ValueError('Coordinates outside image')
    for i,status in enumerate(labels):
        if status in ('occluded','missing') and finite[i]:
            raise ValueError('Occluded/missing landmark must have no coordinates')
        if status in ('manual_visible','ai','interpolated') and not finite[i]:
            raise ValueError('Visible landmark must have finite coordinates')
    edits[frame]=arr.copy()
    statuses[frame]=list(labels)
    save_session()
    return {'saved_frame':frame,'state_file':str(state_file)}
output.register_callback('fingerball2h.save_frame',receive_annotation)
def make_data_url(f):
    img=get_video_frame(f)
    bgr=cv2.cvtColor(img,cv2.COLOR_RGB2BGR)
    ok,data=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,90])
    if not ok:raise RuntimeError('Could not encode frame '+str(f))
    return 'data:image/jpeg;base64,'+base64.b64encode(data).decode('ascii')
frames_json={str(f):{
    'src':make_data_url(f),
    'coords':[[None if not np.isfinite(a) else float(a) for a in xy] for xy in edits[f]],
    'statuses':list(statuses[f])} for f in FRAME_IDS}
html_page=r"""<div id="fb2h_root" style="font:14px Arial,sans-serif;background:#0b1220;color:#f8fafc;padding:14px;border-radius:12px">
<div style="display:flex;flex-wrap:wrap;align-items:center;gap:10px;margin-bottom:10px">
<label>Frame <select id="fb2h_frame"></select></label>
<label>Landmark <select id="fb2h_point"></select></label>
<label>拡大範囲 <input id="fb2h_crop" type="range" min="100" max="750" step="25" value="300"><span id="fb2h_cropval">300 px</span></label>
<button id="fb2h_place">選択点をクリック配置</button>
<button id="fb2h_occlude">遮蔽・不明</button>
<button id="fb2h_restore">AI位置に戻す</button>
<button id="fb2h_undo">Undo</button>
<button id="fb2h_save" style="background:#047857;color:white;padding:7px">Driveに保存</button>
</div>
<canvas id="fb2h_canvas" width="800" height="800" style="display:block;max-width:100%;touch-action:none;border:1px solid #475569;border-radius:6px;cursor:crosshair"></canvas>
<div id="fb2h_message" style="margin-top:10px;color:#a5f3fc">画像を読み込み中…</div>
<div style="font-size:12px;color:#cbd5e1;margin-top:8px">灰色：AI推定／橙：手動修正／青：補間。指名は編集対象のみ表示。遮蔽点は描画せずCSVへ欠損として保存します。</div>
</div>
<script>
(function(){
const DATA=__DATA__;
const ORIGINAL=__ORIGINAL__;
const NAMES=__NAMES__, FINGERS=__FINGERS__, COLORS=__COLORS__;
const WIDTH=__WIDTH__,HEIGHT=__HEIGHT__,INITIAL_FRAME=__INITIAL_FRAME__;
const root=document.getElementById('fb2h_root');
const frameSelect=root.querySelector('#fb2h_frame'),pointSelect=root.querySelector('#fb2h_point');
const range=root.querySelector('#fb2h_crop'),cv=root.querySelector('#fb2h_canvas');
const ctx=cv.getContext('2d'),message=root.querySelector('#fb2h_message');
const keys=Object.keys(DATA).map(Number).sort((a,b)=>a-b);
let current=String(keys.includes(INITIAL_FRAME)?INITIAL_FRAME:keys[0]);
let selected=8,img=null,viewport={x0:0,y0:0,side:300};
let drag=-1,place=false,history=[],dirty=false,loading=false;
const copy=o=>JSON.parse(JSON.stringify(o));
const valid=p=>p && p.length===2 && p[0]!==null && p[1]!==null;
const status=txt=>{message.textContent=txt;};
keys.forEach(f=>{let o=document.createElement('option');o.value=String(f);o.textContent=String(f);frameSelect.appendChild(o)});
NAMES.forEach((name,i)=>{let o=document.createElement('option');o.value=String(i);o.textContent=i+' '+name;pointSelect.appendChild(o)});
frameSelect.value=current;pointSelect.value=String(selected);
let activeImage=null;
function loadImage(){
 let target=current;let next=new Image();
 loading=true;status('Frame '+target+' 読込中');
 next.onload=()=>{if(current!==target)return;img=next;loading=false;viewportUpdate();draw();status('Frame '+target+'：ドラッグで修正できます')};
 next.onerror=()=>{loading=false;status('画像を読み込めませんでした')};
 next.src=DATA[current].src;activeImage=next;
}
function center(){
 const pts=DATA[current].coords.filter(valid);
 const xs=pts.map(p=>p[0]).sort((a,b)=>a-b),ys=pts.map(p=>p[1]).sort((a,b)=>a-b);
 if(xs.length){return [xs[Math.floor(xs.length/2)],ys[Math.floor(ys.length/2)]];}
 return [WIDTH/2,HEIGHT/2];
}
function viewportUpdate(){
 let [cx,cy]=center(),side=Number(range.value);
 side=Math.min(side,Math.max(WIDTH,HEIGHT));
 viewport={side:side,x0:Math.max(0,Math.min(WIDTH-side,cx-side/2)),y0:Math.max(0,Math.min(HEIGHT-side,cy-side/2))};
 if(WIDTH<side)viewport.x0=0;
 if(HEIGHT<side)viewport.y0=0;
}
function screen(p){return [(p[0]-viewport.x0)*cv.width/viewport.side,(p[1]-viewport.y0)*cv.height/viewport.side]}
function imagePosition(e){
 const r=cv.getBoundingClientRect(),px=(e.clientX-r.left)*cv.width/r.width,py=(e.clientY-r.top)*cv.height/r.height;
 return [Math.max(0,Math.min(WIDTH-1,viewport.x0+px*viewport.side/cv.width)),
         Math.max(0,Math.min(HEIGHT-1,viewport.y0+py*viewport.side/cv.height))];
}
function draw(){
 if(!img)return;
 ctx.clearRect(0,0,cv.width,cv.height);
 ctx.drawImage(img,viewport.x0,viewport.y0,
              Math.min(viewport.side,WIDTH-viewport.x0),Math.min(viewport.side,HEIGHT-viewport.y0),
              0,0,cv.width*Math.min(viewport.side,WIDTH-viewport.x0)/viewport.side,
              cv.height*Math.min(viewport.side,HEIGHT-viewport.y0)/viewport.side);
 const a=DATA[current].coords;
 Object.entries(FINGERS).forEach(([name,ids])=>{
  ctx.strokeStyle=COLORS[name];ctx.lineWidth=2.5;
  for(let j=0;j<ids.length-1;j++){
   const p=a[ids[j]],q=a[ids[j+1]];if(!valid(p)||!valid(q))continue;
   const v=screen(p),w=screen(q);ctx.beginPath();ctx.moveTo(v[0],v[1]);ctx.lineTo(w[0],w[1]);ctx.stroke();
  }
 });
 a.forEach((p,i)=>{
  if(!valid(p))return;
  const v=screen(p),s=DATA[current].statuses[i];
  ctx.beginPath();ctx.arc(v[0],v[1],i===selected?6:3.3,0,Math.PI*2);
  ctx.fillStyle=s==='manual_visible'?'#f59e0b':s==='interpolated'?'#67e8f9':'#e5e7eb';
  ctx.fill();if(i===selected){ctx.strokeStyle='#0f172a';ctx.lineWidth=1.5;ctx.stroke();
    ctx.font='15px Arial';ctx.fillStyle='#fde047';ctx.fillText(i+' '+NAMES[i],v[0]+10,v[1]-12);}
 });
 range.parentElement.querySelector('#fb2h_cropval').textContent=range.value+' px';
}
function snapshot(){history.push({frame:current,data:copy(DATA[current])});if(history.length>60)history.shift()}
async function saveFrame(){
 const data=DATA[current],f=Number(current);
 if(!window.google||!google.colab||!google.colab.kernel){
  status('Colab通信がありません。Google Colab内で開いてください');return false;
 }
 try{
  status('Drive保存中...');
  await google.colab.kernel.invokeFunction('fingerball2h.save_frame',
   [{frame:f,coords:data.coords,statuses:data.statuses}],{});
  dirty=false;status('Frame '+f+' をDriveへ保存しました');return true;
 }catch(e){status('保存エラー: '+e);return false;}
}
function updatePoint(i,pos,kind){snapshot();DATA[current].coords[i]=pos;DATA[current].statuses[i]=kind;dirty=true;draw()}
function setFrame(f){
 const change=async()=>{current=String(f);frameSelect.value=current;img=null;loadImage();};
 if(dirty){saveFrame().then(ok=>{if(ok)change();else frameSelect.value=current;});}
 else change();
}
frameSelect.addEventListener('change',e=>setFrame(e.target.value));
pointSelect.addEventListener('change',e=>{selected=Number(e.target.value);draw()});
range.addEventListener('input',()=>{viewportUpdate();draw()});
cv.addEventListener('pointerdown',e=>{
 if(loading||!img)return;const p=imagePosition(e);
 if(place){place=false;root.querySelector('#fb2h_place').style.background='';
   updatePoint(selected,p,'manual_visible');status(NAMES[selected]+' の点を設置');return;}
 const a=DATA[current].coords;
 const d=a.map(q=>valid(q)?Math.hypot(q[0]-p[0],q[1]-p[1]):Infinity);
 let i=d.indexOf(Math.min(...d));
 if(d[i]>Math.max(12,viewport.side*.08)){status('点から遠いです。対象点を選び「クリック配置」を使ってください');return;}
 selected=i;pointSelect.value=String(i);snapshot();drag=i;
 try{cv.setPointerCapture(e.pointerId)}catch(error){}
});
cv.addEventListener('pointermove',e=>{
 if(drag<0)return;DATA[current].coords[drag]=imagePosition(e);
 DATA[current].statuses[drag]='manual_visible';dirty=true;draw();
});
cv.addEventListener('pointerup',e=>{if(drag<0)return;drag=-1;status('変更しました。Driveに保存してください');});
cv.addEventListener('pointercancel',()=>{drag=-1});
root.querySelector('#fb2h_place').addEventListener('click',e=>{
 place=true;e.target.style.background='#f59e0b';status(NAMES[selected]+' を配置する位置をクリックしてください');
});
root.querySelector('#fb2h_occlude').addEventListener('click',()=>{
 updatePoint(selected,[null,null],'occluded');status('遮蔽・不明に設定');
});
root.querySelector('#fb2h_restore').addEventListener('click',()=>{
 const p=copy(ORIGINAL[current][selected]),kind=valid(p)?'ai':'missing';
 updatePoint(selected,p,kind);status('元のAI点に戻しました');
});
root.querySelector('#fb2h_undo').addEventListener('click',()=>{
 if(!history.length){status('Undo履歴がありません');return;}
 const h=history.pop();
 if(h.frame!==current){history.push(h);status('Undoは同じフレーム内で操作してください');return;}
 DATA[current]=h.data;dirty=true;draw();status('Undoしました');
});
root.querySelector('#fb2h_save').addEventListener('click',saveFrame);
window.addEventListener('beforeunload',e=>{if(dirty){e.preventDefault();e.returnValue='';}});
loadImage();
})();</script>""";
html_page=html_page.replace('__DATA__',json.dumps(frames_json,ensure_ascii=False))
html_page=html_page.replace('__ORIGINAL__',json.dumps({
  str(f):[[None if not np.isfinite(x) else float(x) for x in pt] for pt in original[f]]
  for f in FRAME_IDS}))
html_page=html_page.replace('__NAMES__',json.dumps(NAMES))
html_page=html_page.replace('__FINGERS__',json.dumps(FINGERS))
html_page=html_page.replace('__COLORS__',json.dumps(COLORS))
html_page=html_page.replace('__WIDTH__',str(W)).replace('__HEIGHT__',str(H))
html_page=html_page.replace('__INITIAL_FRAME__',str(190 if 190 in FRAME_IDS else FRAME_IDS[0]))
display(HTML(html_page))
print('HTML Canvasアノテータを表示しました。編集後はDrive保存ボタンを押してください。')

## 4. 編集品質の確認と保存

**手動修正（manual_visible）・遮蔽（occluded）・AI初期推定（ai）を区別**し、元のZ値は触りません。エクスポート後のCSVは2D座標の解析用であり、Stage 2Gの疑似3Dに無条件で投入しないでください。

In [ ]:
save_session()
audit=[]
for f in FRAME_IDS:
    for i in range(21):
        a=original[f][i];b=edits[f][i]
        shift=float(np.linalg.norm(a-b)) if np.isfinite(a).all() and np.isfinite(b).all() else np.nan
        audit.append({'frame':f,'landmark_id':i,'landmark_name':NAMES[i],
                      'status':statuses[f][i],'ai_x_px':a[0],'ai_y_px':a[1],
                      'edited_x_px':b[0],'edited_y_px':b[1],
                      'correction_distance_px':shift})
audit_df=pd.DataFrame(audit)
audit_file=out_dir/'hand_landmarks_manual_audit.csv'
audit_df.to_csv(audit_file,index=False)
corrected=source_df.copy()
corrected['annotation_status']='outside_edit_window'
corrected['xy_source']='original_media_pipe'
corrected['z_source']='original_media_pipe_unmodified'
for f in FRAME_IDS:
    for i in range(21):
        mask=(corrected.frame==f)&(corrected.landmark_id==i)
        p=edits[f][i];st=statuses[f][i]
        corrected.loc[mask,'annotation_status']=st
        corrected.loc[mask,'xy_source']=('manual_2d' if st=='manual_visible'
             else 'interpolated_2d' if st=='interpolated'
             else 'unobserved' if st in ('occluded','missing') else 'original_media_pipe')
        corrected.loc[mask,'x_px']=p[0]
        corrected.loc[mask,'y_px']=p[1]
        if 'x_norm' in corrected.columns:corrected.loc[mask,'x_norm']=p[0]/W
        if 'y_norm' in corrected.columns:corrected.loc[mask,'y_norm']=p[1]/H
        if st in ('occluded','missing'):
            corrected.loc[mask,'detected']=0
            corrected.loc[mask,'z_relative']=np.nan
            corrected.loc[mask,'z_source']='unobserved'
corrected_file=out_dir/'hand_21_landmarks_2d_corrected.csv'
corrected.to_csv(corrected_file,index=False)
print('Saved:',audit_file)
print('Saved:',corrected_file)
print('Statuses:',audit_df.status.value_counts().to_dict())
print('Manual shift mean (pixels):',
  audit_df.loc[audit_df.status=='manual_visible','correction_distance_px'].mean())

## 5. 任意：キーフレーム間の補間（初期値としてのみ）

**下のセルは実行した時点で補間を書き込みます。** 手動点が両端にあり、途中点がAI推定のままの区間だけ補間します。**遮蔽・不明と指定した点、評価用に残すフレーム、手動修正済みの点には上書きしません。** 補間値は正解ではありません。比較用の参考値として使用してください。

In [ ]:
ENABLE_INTERPOLATION=False   # 補間する場合のみTrueへ変更して実行
HOLDOUT_FRAMES=[188,192]    # 補正・補間に使用しない評価用フレーム
if ENABLE_INTERPOLATION:
    count=0
    for i in range(21):
        anchors=[f for f in FRAME_IDS if statuses[f][i]=='manual_visible' and
                 np.isfinite(edits[f][i]).all() and f not in HOLDOUT_FRAMES]
        for a,b in zip(anchors[:-1],anchors[1:]):
            if b<=a+1:continue
            for f in range(a+1,b):
                if f in HOLDOUT_FRAMES or statuses[f][i]!='ai':continue
                take_a=edits[a][i];take_b=edits[b][i]
                t=(f-a)/(b-a)
                edits[f][i]=(1-t)*take_a+t*take_b
                statuses[f][i]='interpolated'
                count+=1
    save_session()
    print('Interpolated landmark points:',count)
    print('2D補正CSVを更新するには「4. 編集品質の確認と保存」を再実行してください')
else:
    print('補間は無効です。必要な場合だけENABLE_INTERPOLATION=Trueにしてください')

## 6. 画像比較・共有用ZIP（元推定と編集後の差を可視化）

In [ ]:
def comparison_png(frame_num):
    rgb=get_video_frame(frame_num)
    left=rgb.copy();right=rgb.copy()
    for target,coords,draw_color in ((left,original[frame_num],(170,230,255)),
                                      (right,edits[frame_num],(90,235,110))):
        for finger,idxs in FINGERS.items():
            for a,b in zip(idxs[:-1],idxs[1:]):
                if np.isfinite(coords[[a,b]]).all():
                    cv2.line(target,tuple(np.rint(coords[a]).astype(int)),
                             tuple(np.rint(coords[b]).astype(int)),draw_color,2)
        for p in coords:
            if np.isfinite(p).all():
                cv2.circle(target,tuple(np.rint(p).astype(int)),3,draw_color,-1)
    cx,cy=center_for_frame(frame_num)
    side=min(CROP_SIZE,max(W,H))
    x0=int(np.clip(round(cx-side/2),0,max(W-1,0)))
    y0=int(np.clip(round(cy-side/2),0,max(H-1,0)))
    x1=min(W,x0+side);y1=min(H,y0+side)
    L=cv2.resize(left[y0:y1,x0:x1],(640,640))
    R=cv2.resize(right[y0:y1,x0:x1],(640,640))
    canvas=np.concatenate([L,R],axis=1)
    for x,title in [(12,'MediaPipe original'),(652,'Manual-guided 2D')]:
        cv2.putText(canvas,title,(x,32),cv2.FONT_HERSHEY_SIMPLEX,
                    .75,(255,255,255),2,cv2.LINE_AA)
    cv2.putText(canvas,f'Frame {frame_num}',(12,624),
                cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
    return canvas
from PIL import Image
comparison_paths=[]
for f in KEY_FRAMES:
    if f not in FRAME_IDS:continue
    im=comparison_png(f)
    p=out_dir/f'frame_{f}_original_vs_manual.png'
    Image.fromarray(im).save(p)
    comparison_paths.append(p)
report=out_dir/'stage2h_annotation_report.json'
report.write_text(json.dumps({
 'video':str(video_path),'source_csv':str(csv_path),'frame_window':[FIRST_FRAME,LAST_FRAME],
 'hand_manual_label':THROWING_HAND,'key_frame_candidates':KEY_FRAMES,
 'holdout_recommended':[188,192],
 'annotations':{str(f):{
    'manual_visible':int(statuses[f].count('manual_visible')),
    'occluded':int(statuses[f].count('occluded')),
    'interpolated':int(statuses[f].count('interpolated'))}
    for f in FRAME_IDS},
 'limitations':['Manual point editing only changes 2D XY; Z is not corrected',
                'Interpolated positions are estimates, not ground truth',
                'Do not use manually corrected XY with old Z as validated 3D data']
},ensure_ascii=False,indent=2),encoding='utf-8')
package=out_dir/'FingerBall_Stage2H_manual_annotation_package.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for item in [state_file,audit_file,corrected_file,report]+comparison_paths:
        if item.is_file():z.write(item,arcname=item.name)
print('Saved comparisons:',len(comparison_paths))
print('Package:',package)
print('注：編集後に4のCSV出力セルを再実行してからZIPを作成してください')

## 次の段階への橋渡し

手動点を6フレーム修正したら、まず**補間を実行せず**に保存した座標と元動画をレビューしてください。数値評価には独立の手動評価フレームを使い、**修正対象と評価対象を混同しない**ようにしてください。Stage 2I（MANO等のメッシュフィット）はこの2D修正CSVと可視性フラグを入力に用いる想定です。未修正の`z_relative`は3Dの正解ではありません。